# Fruit Object Recognition by OpenCV

## Install OpenCV library


In [ ]:
pip install opencv-python

## Fruit Object Recognition




In [1]:
import cv2

img = cv2.imread("Fruit.webp")

if img is None:
    print("Image not found!")
    exit()

img = cv2.resize(img, (800, 420))
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Fruits = high saturation (remove pink background)
mask = cv2.inRange(hsv, (0, 40, 40), (179, 255, 255))
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print("Fruits:", len(contours))

for c in contours:
    if cv2.contourArea(c) < 2000:
        continue

    x, y, w, h = cv2.boundingRect(c)
    aspect = max(w, h) / min(w, h)

   
    fruit_mask = cv2.bitwise_xor(mask, mask)  
    cv2.drawContours(fruit_mask, [c], -1, 255, -1)
    B, G, R, _ = cv2.mean(img, mask=fruit_mask)

    
    roi_hsv = hsv[y:y + h, x:x + w]
    roi_bgr = img[y:y + h, x:x + w]
    roi_m = fruit_mask[y:y + h, x:x + w]

    green = cv2.inRange(roi_hsv, (28, 40, 40), (95, 255, 255))
    green = cv2.bitwise_and(green, roi_m)

    
    _, g_ch, r_ch = cv2.split(roi_bgr)
    greener = cv2.threshold(cv2.subtract(g_ch, r_ch), 10, 255, cv2.THRESH_BINARY)[1]
    green = cv2.bitwise_and(green, greener)

    green_ratio = cv2.countNonZero(green) / max(1, cv2.countNonZero(roi_m))


    if aspect >= 1.3:
        label = "Banana"
    elif green_ratio > 0.015:
        label = "Watermelon"
    elif B > 90:
        label = "Grapefruit"
    else:
        label = "Orange"

    print(label)

    cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(img, label, (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

cv2.imshow("Fruit Object Recognition", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


Fruits: 4
Orange
Grapefruit
Watermelon
Banana
